**1. ติดตั้งและเชื่อมต่อ Neo4j online server**

In [ ]:
!pip install -q neo4j


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 6.1 MB/s eta 0:00:00


In [ ]:
import neo4j

print("Neo4j Python Driver Version:", neo4j.__version__)


Neo4j Python Driver Version: 6.3.1


In [ ]:
from getpass import getpass
from neo4j import GraphDatabase

URI = "neo4j+s://4e8333b1.databases.neo4j.io"
USERNAME = "4e8333b1"
PASSWORD = getpass("Password: ")

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

driver.verify_connectivity()

print("Connected successfully")


Password: ··········
Connected successfully


หา home database แล้วเก็บชื่อไว้ใช้กับทุก Query

In [ ]:
result = driver.execute_query(
    "SHOW HOME DATABASE"
)

DATABASE = result.records[0]["name"]

print("Database =", DATABASE)


Database = 4e8333b1


**2. ล้างข้อมูลผลไม้เดิม (ถ้ามี)**

ลบเฉพาะ Node `User` และ `Fruit` ของโปรเจกต์นี้ เพื่อให้รันซ้ำได้

In [ ]:
driver.execute_query(
    """
    MATCH (n)
    WHERE n:User OR n:Fruit
    DETACH DELETE n
    """,
    database_=DATABASE
)

print("ล้างข้อมูลเดิมเรียบร้อย")


ล้างข้อมูลเดิมเรียบร้อย


**3. สร้าง Constraint** (กันชื่อซ้ำ และทำให้ MATCH เร็วขึ้น)

In [ ]:
driver.execute_query(
    """
    CREATE CONSTRAINT user_name_unique
    IF NOT EXISTS
    FOR (u:User)
    REQUIRE u.name IS UNIQUE
    """,
    database_=DATABASE
)

driver.execute_query(
    """
    CREATE CONSTRAINT fruit_name_unique
    IF NOT EXISTS
    FOR (f:Fruit)
    REQUIRE f.name IS UNIQUE
    """,
    database_=DATABASE
)

print("User / Fruit constraint created")


User / Fruit constraint created


**4. add user** — รายชื่อผู้ใช้ทั้ง 10 คน

In [ ]:
users = [
    "Ananda", "Boonchit", "Chutima", "Danai", "boy",
    "yu", "pond", "pat", "poo", "ya"
]

print("จำนวน User =", len(users))


จำนวน User = 10


In [ ]:
result = driver.execute_query(
    """
    UNWIND $users AS name
    MERGE (u:User {name: name})
    RETURN count(u) AS total
    """,
    users=users,
    database_=DATABASE
)

print("จำนวน User ที่ประมวลผล =", result.records[0]["total"])


จำนวน User ที่ประมวลผล = 10


**5. add fruits** — รายการผลไม้ทั้ง 5 ชนิด

In [ ]:
fruits = ["Mango", "Apple", "Durian", "Banana", "Grape"]

print("จำนวน Fruit =", len(fruits))


จำนวน Fruit = 5


In [ ]:
result = driver.execute_query(
    """
    UNWIND $fruits AS name
    MERGE (f:Fruit {name: name})
    RETURN count(f) AS total
    """,
    fruits=fruits,
    database_=DATABASE
)

print("จำนวน Fruit ที่ประมวลผล =", result.records[0]["total"])


จำนวน Fruit ที่ประมวลผล = 5


**6. สร้าง Relationship LIKES** (ผู้ใช้ชอบผลไม้นี้)

In [ ]:
likes = [
    ("Ananda", "Mango"), ("Ananda", "Apple"),
    ("Boonchit", "Mango"), ("Boonchit", "Apple"), ("Boonchit", "Banana"),
    ("Chutima", "Mango"), ("Chutima", "Durian"),
    ("Danai", "Banana"), ("Danai", "Grape"),
    ("boy", "Durian"), ("boy", "Grape"),
    ("yu", "Mango"), ("yu", "Banana"),
    ("pond", "Apple"), ("pond", "Durian"),
    ("pat", "Grape"), ("pat", "Mango"),
    ("poo", "Banana"), ("poo", "Apple"),
    ("ya", "Durian"), ("ya", "Mango")
]

print("จำนวน LIKES =", len(likes))


จำนวน LIKES = 21


In [ ]:
likes_data = [
    {"user": u, "fruit": f}
    for u, f in likes
]

driver.execute_query(
    """
    UNWIND $likes AS row
    MATCH (u:User {name: row.user}), (f:Fruit {name: row.fruit})
    MERGE (u)-[:LIKES]->(f)
    """,
    likes=likes_data,
    database_=DATABASE
)

print("สร้าง LIKES สำเร็จ")


สร้าง LIKES สำเร็จ


**7. สร้าง Relationship SIMILAR_TO** (User ที่มีความชอบผลไม้ร่วมกัน)

ใช้ `SIMILAR_TO` แทน `FRIEND_OF` โดยเชื่อม User ที่ชอบผลไม้ชนิดเดียวกันอย่างน้อย 1 ชนิด

In [ ]:
driver.execute_query(
    """
    MATCH (u1:User)-[:LIKES]->(f:Fruit)<-[:LIKES]-(u2:User)
    WHERE u1 <> u2
      AND u1.name < u2.name
    MERGE (u1)-[:SIMILAR_TO]->(u2)
    """,
    database_=DATABASE
)

print("สร้าง SIMILAR_TO สำเร็จ")


สร้าง SIMILAR_TO สำเร็จ


In [ ]:
result = driver.execute_query(
    """
    MATCH (u1:User)-[:SIMILAR_TO]->(u2:User)
    RETURN u1.name AS User1, u2.name AS User2
    ORDER BY User1, User2
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["User1"], "<->", record["User2"])


Ananda <-> Boonchit
Ananda <-> Chutima
Ananda <-> pat
Ananda <-> pond
Ananda <-> poo
Ananda <-> ya
Ananda <-> yu
Boonchit <-> Chutima
Boonchit <-> Danai
Boonchit <-> pat
Boonchit <-> pond
Boonchit <-> poo
Boonchit <-> ya
Boonchit <-> yu
Chutima <-> boy
Chutima <-> pat
Chutima <-> pond
Chutima <-> ya
Chutima <-> yu
Danai <-> boy
Danai <-> pat
Danai <-> poo
Danai <-> yu
boy <-> pat
boy <-> pond
boy <-> ya
pat <-> ya
pat <-> yu
pond <-> poo
pond <-> ya
poo <-> yu
ya <-> yu


ตรวจดูขนาดของ Graph (ควรได้ 15 Node, 21 Relationship)

In [ ]:
result = driver.execute_query(
    "MATCH (n) RETURN count(n) AS total_nodes",
    database_=DATABASE
)
print("จำนวน Node =", result.records[0]["total_nodes"])

result = driver.execute_query(
    "MATCH ()-[r]->() RETURN count(r) AS total_relationships",
    database_=DATABASE
)
print("จำนวน Relationship ทั้งหมด =", result.records[0]["total_relationships"])

result = driver.execute_query(
    "MATCH ()-[r:LIKES]->() RETURN count(r) AS total_likes",
    database_=DATABASE
)
print("จำนวน LIKES =", result.records[0]["total_likes"])

result = driver.execute_query(
    "MATCH ()-[r:SIMILAR_TO]->() RETURN count(r) AS total_similar",
    database_=DATABASE
)
print("จำนวน SIMILAR_TO =", result.records[0]["total_similar"])


จำนวน Node = 15
จำนวน Relationship ทั้งหมด = 53
จำนวน LIKES = 21
จำนวน SIMILAR_TO = 32


**9. ตรวจสอบข้อมูลใน Graph**

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)-[:LIKES]->(f:Fruit)
    RETURN u.name AS User, f.name AS Fruit
    ORDER BY User, Fruit
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["User"], "->", record["Fruit"])


Ananda -> Apple
Ananda -> Mango
Boonchit -> Apple
Boonchit -> Banana
Boonchit -> Mango
Chutima -> Durian
Chutima -> Mango
Danai -> Banana
Danai -> Grape
boy -> Durian
boy -> Grape
pat -> Grape
pat -> Mango
pond -> Apple
pond -> Durian
poo -> Apple
poo -> Banana
ya -> Durian
ya -> Mango
yu -> Banana
yu -> Mango


**10. คำถามที่ 1: Ananda ชอบผลไม้ชนิดอะไรบ้าง?**

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User {name: "Ananda"})-[:LIKES]->(f:Fruit)
    RETURN f.name AS Fruit
    ORDER BY Fruit
    """,
    database_=DATABASE
)

for record in result.records:
    print("Fruit:", record["Fruit"])


Fruit: Apple
Fruit: Mango


ตรงนี้อธิบายได้ว่า Neighbor ของ Ananda คือ Fruit Node ที่เชื่อมกับ Ananda ด้วยความสัมพันธ์ LIKES

**11. คำถามที่ 2: หา User ที่ชอบผลไม้เหมือน Ananda**

In [ ]:
result = driver.execute_query(
    """
    MATCH
        (target:User {name: "Ananda"})
        -[:SIMILAR_TO]->
        (similar:User)
        -[:LIKES]->
        (recommended:Fruit)
    WHERE NOT (target)-[:LIKES]->(recommended)
    RETURN recommended.name AS Fruit, count(DISTINCT similar) AS Score
    ORDER BY Score DESC, Fruit
    """,
    database_=DATABASE
)

print("Recommendation:")
for record in result.records:
    print(record["Fruit"], "score =", record["Score"])


Recommendation:
Banana score = 3
Durian score = 3
Grape score = 1


Ananda ชอบ Mango และ Apple ดังนั้นเราสามารถหา User คนอื่นที่ชอบ Mango หรือ Apple เหมือน Ananda ได้

def recommend_fruits(driver, database, target_user):
    result = driver.execute_query(
        """
        MATCH
            (target:User {name: $target_user})
            -[:SIMILAR_TO]->
            (similar:User)
            -[:LIKES]->
            (recommended:Fruit)

        WHERE NOT EXISTS {
            MATCH (target)-[:LIKES]->(recommended)
        }

        RETURN
            recommended.name AS Fruit,
            count(DISTINCT similar) AS Score

        ORDER BY Score DESC, Fruit
        """,
        target_user=target_user,
        database_=database
    )

    return [(record["Fruit"], record["Score"]) for record in result.records]


In [ ]:
result = driver.execute_query(
    """
    MATCH
        (target:User {name: "Ananda"})-[:LIKES]->(liked:Fruit)
        <-[:LIKES]-(similar:User)-[:LIKES]->(recommended:Fruit)
    WHERE NOT (target)-[:LIKES]->(recommended)
    RETURN recommended.name AS Fruit, count(DISTINCT similar) AS Score
    ORDER BY Score DESC, Fruit
    """,
    database_=DATABASE
)

print("Recommendation:")
for record in result.records:
    print(record["Fruit"], "score =", record["Score"])


Recommendation:
Banana score = 3
Durian score = 3
Grape score = 1


**13. Recommendation ที่เขียนเป็นฟังก์ชัน**

In [ ]:
def recommend_fruits(driver, database, target_user):
    result = driver.execute_query(
        """
        MATCH
            (target:User {name: $target_user})-[:LIKES]->(liked:Fruit)
            <-[:LIKES]-(similar:User)-[:LIKES]->(recommended:Fruit)
        WHERE NOT (target)-[:LIKES]->(recommended)
        RETURN recommended.name AS Fruit, count(DISTINCT similar) AS Score
        ORDER BY Score DESC, Fruit
        """,
        target_user=target_user,
        database_=database
    )

    return [(record["Fruit"], record["Score"]) for record in result.records]


เรียกใช้งาน Recommendation สำหรับ Ananda

In [ ]:
result = recommend_fruits(driver, DATABASE, "Ananda")
print(result)


[('Banana', 3), ('Durian', 3), ('Grape', 1)]


เรียกใช้งาน Recommendation สำหรับ Chutima

In [ ]:
result = recommend_fruits(driver, DATABASE, "Chutima")
print(result)


[('Apple', 3), ('Banana', 2), ('Grape', 2)]


เรียกใช้งาน Recommendation สำหรับ Boonchit

In [ ]:
result = recommend_fruits(driver, DATABASE, "Boonchit")
print(result)


[('Durian', 3), ('Grape', 2)]


**14. ตรวจสอบผลไม้ที่ User เลือก**

In [ ]:
target_user = "Ananda"

result = driver.execute_query(
    """
    MATCH (u:User {name: $target_user})-[:LIKES]->(f:Fruit)
    RETURN f.name AS Fruit
    ORDER BY Fruit
    """,
    target_user=target_user,
    database_=DATABASE
)

ananda_fruits = [record["Fruit"] for record in result.records]

print(ananda_fruits)


['Apple', 'Mango']


**15. สรุป Graph**

In [ ]:
result = driver.execute_query(
    """
    MATCH (u:User)-[:LIKES]->(f:Fruit)
    RETURN u.name AS User, collect(f.name) AS Fruits
    ORDER BY User
    """,
    database_=DATABASE
)

for record in result.records:
    print(record["User"], ":", ", ".join(record["Fruits"]))


Ananda : Mango, Apple
Boonchit : Mango, Apple, Banana
Chutima : Mango, Durian
Danai : Banana, Grape
boy : Durian, Grape
pat : Mango, Grape
pond : Apple, Durian
poo : Apple, Banana
ya : Mango, Durian
yu : Mango, Banana


**สรุป**

Recommender System แบบ Graph ใช้ความสัมพันธ์ `LIKES` เพื่อเก็บความชอบของ User และใช้ `SIMILAR_TO` เพื่อเชื่อม User ที่มีความชอบผลไม้ร่วมกัน

ระบบจะค้นหา User ที่ `SIMILAR_TO` กับ User เป้าหมาย แล้วนำผลไม้ที่ User เหล่านั้นชอบ แต่ User เป้าหมายยังไม่ได้ชอบ มาใช้เป็นคำแนะนำ โดยนับจำนวน User ที่คล้ายกันเป็น Score

**16. ปิดการเชื่อมต่อ Neo4j**

In [ ]:
driver.close()
print("ปิดการเชื่อมต่อ Neo4j เรียบร้อย")
